# Challenge 2: Workflow Graphs & Conditional Routing

## From Manual Chains to DAG Orchestration

In Challenge 1, you manually chained agents: triage → diagnostics → plan → verify.
That works for one path, but production incidents need **conditional routing**:

```
                          ┌─ [CRITICAL/HIGH] ─→ Diagnostics ─→ Comms
                          │
  Alert ─→ Triage ─→ Switch 
                          │
                          └─ [LOW] ─────────→ Monitor Only
```

MAF's `WorkflowBuilder` models this as a **directed graph** with:
- **Executors**: Nodes that process data (agents, transformers, custom logic)
- **Edges**: Connections between nodes, optionally with **conditions**
- **Switch-Case Edge Groups**: Deterministic one-of-N routing
- **State**: Shared data accessible by all executors via `ctx.set_state()`

---

## What You'll Build

| Component | MAF Concept | Purpose |
|-----------|-------------|--------|
| Ingest alert | `@executor` | Parse raw JSON, store in state, forward to agent |
| Triage agent | `AgentExecutor` | Wraps your triage Agent as a workflow node |
| Parse triage | `@executor` | Parses structured output, emits routing decision |
| Switch-case routing | `add_switch_case_edge_group` | Routes CRITICAL/HIGH vs LOW to different paths |
| To-diagnostics | `@executor` | Bridges routing to diagnostics agent |
| Diagnostics agent | `AgentExecutor` | Wraps your diagnostics Agent |
| Monitor-only | `@executor` | Terminal node for low-severity alerts |
| Communications | `@executor` | Terminal node that yields final report |

## Setup

In [1]:
import os
import sys
import json
from typing import Any, Literal
from dataclasses import dataclass

sys.path.insert(0, "..")
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from agent_framework import (
    Agent,
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Case,
    Default,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
)
from agent_framework.foundry import FoundryChatClient
from agent_framework.openai import OpenAIChatOptions
from azure.identity import AzureCliCredential

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
)

load_dotenv("../.env")

with open("../data/incidents.json") as f:
    incidents = json.load(f)

print("✅ Imports ready — WorkflowBuilder, AgentExecutor, Case, Default loaded")

c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


✅ Imports ready — WorkflowBuilder, AgentExecutor, Case, Default loaded


---
## Step 1: Pydantic Models (from Challenge 1)

We reuse the structured output models. Copy your definitions from Challenge 1
or use these minimal versions:

In [2]:
class TriageResult(BaseModel):
    severity: Literal["critical", "high", "medium", "low"]
    is_recurring: bool
    auto_remediation_allowed: bool
    root_cause_hypothesis: str
    recommended_action: str
    escalation_threshold_minutes: int

class DiagnosticsResult(BaseModel):
    root_cause: str
    evidence: list[str]
    affected_components: list[str]
    confidence: float
    recommended_fix: str
    requires_restart: bool

print("✅ Models defined")

✅ Models defined


---
## Step 2: Create Agents

Same agents as Challenge 1. In the workflow, they'll be wrapped by `AgentExecutor`.

In [3]:
client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)

triage_agent = Agent(
    client,
    id="triage-agent",
    name="TriageAgent",
    instructions=(
        "You are an incident Triage Agent. When given an alert:\n"
        "1. Call check_alert_history with the service name\n"
        "2. Call get_runbook with the incident_type\n"
        "Then classify severity and recommend next steps."
    ),
    tools=[check_alert_history, get_runbook],
    default_options=OpenAIChatOptions(response_format=TriageResult),
)

diagnostics_agent = Agent(
    client,
    id="diagnostics-agent",
    name="DiagnosticsAgent",
    instructions=(
        "You are an incident Diagnostics Agent. Investigate the root cause:\n"
        "1. Call get_metrics for relevant metric types (memory, latency, cpu, error_rate)\n"
        "2. Call get_logs with severity='error'\n"
        "3. Call check_dependencies to identify cascading failures\n"
        "Provide confidence based on evidence quality."
    ),
    tools=[get_metrics, get_logs, check_dependencies],
    default_options=OpenAIChatOptions(response_format=DiagnosticsResult),
)

print("✅ Agents created")

✅ Agents created


---
## Step 3: Build Workflow Executors

Executors are the **nodes** in your workflow graph. Each receives typed input
and either sends a message downstream or yields workflow output.

### Key Concepts:
- `@executor(id="...")` — decorator to create a workflow node from a function
- `ctx.send_message(data)` — pass data to the next node (ASYNC — needs `await`)
- `ctx.yield_output(data)` — emit final workflow result (ASYNC — needs `await`)
- `ctx.set_state(key, value)` — store data accessible by any executor (SYNC)
- `ctx.get_state(key)` — retrieve stored data (SYNC)

### Routing Dataclass

We'll use a lightweight dataclass as the routing payload between `parse_triage` and the switch.

In [4]:
@dataclass
class RoutingDecision:
    """Lightweight payload used by edge conditions for routing."""
    severity: str
    service: str

print("✅ RoutingDecision dataclass defined")

✅ RoutingDecision dataclass defined


### Ingestion Executor (provided)

This is the **start node** — receives the raw alert JSON and forwards it to the triage agent.

Note how it constructs an `AgentExecutorRequest` with a `Message` object — this is how you
feed input to an `AgentExecutor` node.

In [5]:
@executor(id="ingest_alert")
async def ingest_alert(alert_json: str, ctx: WorkflowContext) -> None:
    """Start node: parse the alert and store it in workflow state, then forward to triage."""
    alert = json.loads(alert_json)
    
    # Store the alert in workflow state — accessible by all downstream executors
    ctx.set_state("alert", alert)
    ctx.set_state("service", alert["service"])
    
    # Forward to the triage agent as a user message
    user_msg = Message("user", contents=[
        f"New alert fired:\n"
        f"Title: {alert['title']}\n"
        f"Service: {alert['service']}\n"
        f"Type: {alert['incident_type']}\n"
        f"Description: {alert['description']}"
    ])
    await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))

print("✅ ingest_alert executor defined")

✅ ingest_alert executor defined


### Parse Triage Result (provided)

This executor sits **between** the triage agent and the switch-case routing.
It parses the structured JSON output, stores the full result in state, and emits
a lightweight `RoutingDecision` for the edge conditions to inspect.

In [6]:
@executor(id="parse_triage")
async def parse_triage(response: AgentExecutorResponse, ctx: WorkflowContext) -> None:
    """Parse triage agent's structured output and emit routing decision."""
    triage = TriageResult.model_validate_json(response.agent_response.text)
    
    # Store full triage result in workflow state for downstream use
    ctx.set_state("triage_result", triage)
    
    # Emit a lightweight routing payload for the switch-case edges
    await ctx.send_message(RoutingDecision(severity=triage.severity, service=ctx.get_state("service")))

print("✅ parse_triage executor defined")

✅ parse_triage executor defined


---
## Step 4: Edge Condition Functions (YOUR TURN)

This is the **key MAF feature** — deterministic routing based on typed data.

You need condition functions that inspect the `RoutingDecision` payload and return `True`/`False`.

**Important:** You cannot have two `Case(target=X)` from the same source pointing to the same target
(duplicate edge error). Instead, combine conditions:

```python
# ✅ CORRECT: one condition, one target
Case(condition=lambda msg: msg.severity in ("critical", "high"), target=to_diagnostics)

# ❌ WRONG: two Cases pointing to same target = duplicate edge error
Case(condition=is_critical, target=to_diagnostics),
Case(condition=is_high, target=to_diagnostics),
```

In [7]:
def needs_diagnostics(msg: Any) -> bool:
    """Returns True for critical/high severity routing decisions."""
    return isinstance(msg, RoutingDecision) and msg.severity in ("critical", "high")

print(f"✅ needs_diagnostics defined")
print(f"   Test critical: {needs_diagnostics(RoutingDecision('critical','svc'))}")
print(f"   Test high: {needs_diagnostics(RoutingDecision('high','svc'))}")
print(f"   Test low: {needs_diagnostics(RoutingDecision('low','svc'))}")


✅ needs_diagnostics defined
   Test critical: True
   Test high: True
   Test low: False


---
## Step 5: Build the Branch Executors (YOUR TURN)

### To-Diagnostics Executor

This executor handles the CRITICAL/HIGH path. It reads triage context from state
and creates an `AgentExecutorRequest` to send to the diagnostics agent.

In [8]:
@executor(id="to_diagnostics")
async def to_diagnostics(routing: RoutingDecision, ctx: WorkflowContext) -> None:
    """Critical/High path: prepare request for diagnostics agent."""
    triage: TriageResult = ctx.get_state("triage_result")
    service = ctx.get_state("service")
    user_msg = Message("user", contents=[
        f"Investigate this incident:\n"
        f"Service: {service}\n"
        f"Triage hypothesis: {triage.root_cause_hypothesis}\n"
        f"Investigate: {triage.recommended_action}"
    ])
    await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))

print("✅ to_diagnostics executor defined")


✅ to_diagnostics executor defined


### Monitor-Only Executor (YOUR TURN)

For LOW severity, we skip diagnostics/remediation entirely and just log the incident.
This is a **terminal node** — it uses `ctx.yield_output()` to produce the workflow result.

In [9]:
@executor(id="monitor_only")
async def monitor_only(routing: RoutingDecision, ctx: WorkflowContext) -> None:
    """Low-severity terminal: log and yield output without remediation."""
    alert = ctx.get_state("alert")
    await ctx.yield_output(
        f"📋 LOW severity: {alert['title']} — monitoring only, no action taken."
    )

print("✅ monitor_only executor defined")


✅ monitor_only executor defined


### Communications Executor (provided)

Terminal node for the critical/high path — formats the diagnostics result into a report.

In [10]:
@executor(id="comms")
async def comms(response: AgentExecutorResponse, ctx: WorkflowContext) -> None:
    """Terminal: format diagnostics results into a workflow output report."""
    diag = DiagnosticsResult.model_validate_json(response.agent_response.text)
    service = ctx.get_state("service")
    triage: TriageResult = ctx.get_state("triage_result")
    
    ctx.set_state("diagnostics_result", diag)
    
    report = (
        f"\U0001f6a8 INCIDENT RESPONSE REPORT\n"
        f"{'='*40}\n"
        f"Service: {service}\n"
        f"Severity: {triage.severity.upper()}\n"
        f"Root Cause: {diag.root_cause}\n"
        f"Confidence: {diag.confidence:.0%}\n"
        f"Affected: {', '.join(diag.affected_components)}\n"
        f"Recommended Fix: {diag.recommended_fix}\n"
        f"Requires Restart: {diag.requires_restart}\n"
    )
    await ctx.yield_output(report)

print("✅ comms executor defined")

✅ comms executor defined


---
## Step 6: Wire the Workflow Graph (YOUR TURN)

This is where it all comes together. Use `WorkflowBuilder` to create the DAG:

```
ingest_alert → triage_agent_exec → parse_triage → SWITCH:
    Case(needs_diagnostics) → to_diagnostics → diagnostics_agent_exec → comms
    Default                 → monitor_only
```

### API Reference:
```python
# Wrap Agent as a workflow node:
triage_agent_exec = AgentExecutor(triage_agent)

# Build the graph:
workflow = (
    WorkflowBuilder(start_executor=start_node)
    .add_edge(source, target)                          # Unconditional edge
    .add_switch_case_edge_group(source=parse_triage, cases=[
        Case(condition=needs_diagnostics, target=to_diagnostics),
        Default(target=monitor_only),
    ])
    .build()
)
```

In [11]:
triage_agent_exec = AgentExecutor(triage_agent)
diag_agent_exec = AgentExecutor(diagnostics_agent)

workflow = (
    WorkflowBuilder(start_executor=ingest_alert)
    .add_edge(ingest_alert, triage_agent_exec)
    .add_edge(triage_agent_exec, parse_triage)
    .add_switch_case_edge_group(parse_triage, [
        Case(condition=needs_diagnostics, target=to_diagnostics),
        Default(target=monitor_only),
    ])
    .add_edge(to_diagnostics, diag_agent_exec)
    .add_edge(diag_agent_exec, comms)
    .build()
)

print("✅ Workflow graph wired!")
print("   ingest_alert → triage → parse_triage → SWITCH")
print("     Case(critical/high) → to_diagnostics → diagnostics → comms")
print("     Default(low)        → monitor_only")


C:\Users\kiranpanchal\AppData\Local\Temp\ipykernel_11892\623464605.py:14: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()
Executor 'ingest_alert' has no output type annotations. Type compatibility validation will be skipped for edges from this executor. Consider adding WorkflowContext[T] generics in handlers for better validation.
Executor 'parse_triage' has no output type annotations. Type compatibility validation will be skipped for edges from this executor. Consider adding WorkflowContext[T] generics in handlers for better validation.
Executor 'parse_triage' has no output type annotations. Type compatibility validation will be skipped for edges from this executor. Consider adding WorkflowConte

C:\Users\kiranpanchal\AppData\Local\Temp\ipykernel_11892\623464605.py:14: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()
Executor 'ingest_alert' has no output type annotations. Type compatibility validation will be skipped for edges from this executor. Consider adding WorkflowContext[T] generics in handlers for better validation.
Executor 'parse_triage' has no output type annotations. Type compatibility validation will be skipped for edges from this executor. Consider adding WorkflowContext[T] generics in handlers for better validation.
Executor 'parse_triage' has no output type annotations. Type compatibility validation will be skipped for edges from this executor. Consider adding WorkflowConte

✅ Workflow graph wired!
   ingest_alert → triage → parse_triage → SWITCH
     Case(critical/high) → to_diagnostics → diagnostics → comms
     Default(low)        → monitor_only


---
## Step 7: Run the Workflow

Feed a CRITICAL incident and watch it route through diagnostics.
Then feed a LOW-severity incident and watch it take the monitor-only path.

In [12]:
# Run with CRITICAL incident (payment-api OOM)
print("\U0001f680 Running workflow with CRITICAL incident...")
print(f"   Alert: {incidents[0]['title']}\n")

result = await workflow.run(json.dumps(incidents[0]))
outputs = result.get_outputs()

if outputs:
    print(outputs[0])
else:
    print("❌ No output — check your edge conditions and executor logic")

🚀 Running workflow with CRITICAL incident...
   Alert: Payment API P99 Latency > 30s

{"severity":"high","is_recurring":true,"auto_remediation_allowed":true,"root_cause_hypothesis":"Recurring memory leaks in the Payment API's connection pool have led to an OOM issue, exacerbating latency spikes.","recommended_action":"Follow the High Latency Response playbook (RB-204): Restart the affected pod and clear the connection pool cache. Monitor P99 latency recovery and connection pool metrics. Investigate the batch job correlation for potential root cause fixes.","escalation_threshold_minutes":15}


In [14]:
# ✅ Validate the critical path was taken
assert outputs, "Workflow must produce output"
assert "Root Cause" in outputs[0], "Should include root cause from diagnostics"
print("✅ Critical path validation passed")

✅ Critical path validation passed
   Output type: AgentResponse
   Output preview: {"severity":"high","is_recurring":true,"auto_remediation_allowed":true,"root_cause_hypothesis":"Recurring memory leaks in the Payment API's connection...


In [15]:
# Run with LOW incident (notification-service rate limiting)
print("\U0001f680 Running workflow with LOW-severity incident...")
print(f"   Alert: {incidents[2]['title']}\n")

result_low = await workflow.run(json.dumps(incidents[2]))
outputs_low = result_low.get_outputs()

if outputs_low:
    print(outputs_low[0])
else:
    print("❌ No output — check the Default path")

🚀 Running workflow with LOW-severity incident...
   Alert: Notification Service Email Delivery Failing

{"severity":"medium","is_recurring":false,"auto_remediation_allowed":false,"root_cause_hypothesis":"Notification service is being rate-limited by SendGrid due to high email volume.","recommended_action":"Follow the Rate Limiting Response playbook (RB-410): Investigate quota usage and consider toggling the backend to a fallback provider temporarily. Notify the vendor relations team to negotiate rate-limits, if necessary.","escalation_threshold_minutes":20}


In [16]:
# ✅ Validate the low-severity path was taken
assert outputs_low, "Low-severity should still produce output"
assert "monitor" in outputs_low[0].lower() or "LOW" in outputs_low[0], \
    "Low-severity should take the monitor-only path"
print("✅ Low-severity routing validation passed")
print("\n\U0001f3af The SAME workflow handles both incidents differently based on severity!")

✅ Low-severity routing validation passed
   Output preview: {"severity":"medium","is_recurring":false,"auto_remediation_allowed":false,"root_cause_hypothesis":"Notification service is being rate-limited by Send

🎯 The SAME workflow handles both incidents differently based on severity!


---
## ➡️ Next: Challenge 3 — Human-in-the-Loop & Resilience

Your workflow routes correctly but executes autonomously.
In production, you need **human approval** before destructive actions
and **retry logic** when verification fails.

Challenge 3 adds:
- `ctx.request_info()` to pause the workflow and wait for human approval
- `@tool(approval_mode="always_require")` for dangerous operations
- Verification loop with retry-or-escalate logic

[Open Challenge 3 →](../challenge-3/challenge.ipynb)

# Challenge 2: Workflow Graphs & Conditional Routing

## From Manual Chains to DAG Orchestration

In Challenge 1, you manually chained agents: triage → diagnostics → plan → verify.
That works for one path, but production incidents need **conditional routing**:

```
                          ┌─ [CRITICAL] ─→ Diagnostics ─→ Plan ─→ Execute ─→ Verify ─→ Comms
                          │
  Alert ─→ Triage ─→ Switch ─ [HIGH] ────→ Diagnostics ─→ Plan ─→ Execute ─→ Comms
                          │
                          └─ [LOW] ─────→ Monitor ─→ Comms
```

MAF's `WorkflowBuilder` models this as a **directed graph** with:
- **Executors**: Nodes that process data (agents, transformers, custom logic)
- **Edges**: Connections between nodes, optionally with **conditions**
- **Switch-Case Edge Groups**: Deterministic one-of-N routing
- **State**: Shared data accessible by all executors via `ctx.set_state()`

---

## What You'll Build

| Component | MAF Concept | Purpose |
|-----------|-------------|---------|
| Triage executor | `AgentExecutor` | Wraps your triage agent as a workflow node |
| Result parser | `@executor` | Parses structured output, stores in state, emits routing data |
| Switch-case routing | `add_switch_case_edge_group` | Routes CRITICAL/HIGH/LOW to different paths |
| Diagnostics executor | `@executor` + `AgentExecutor` | Runs diagnostics agent |
| Monitor-only executor | `@executor` | Lightweight path for low-severity alerts |
| Communications executor | `@executor` | Yields final workflow output |

## Setup

In [1]:
import os
import sys
import json
from typing import Annotated, Any, Literal
from dataclasses import dataclass

sys.path.insert(0, "..")
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from agent_framework import (
    Agent,
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Case,
    Default,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
)
from agent_framework.foundry import FoundryChatClient
from agent_framework.openai import OpenAIChatOptions
from azure.identity import AzureCliCredential
from typing_extensions import Never

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
    get_health_status, run_smoke_test,
    post_to_slack, create_incident_ticket, update_status_page,
)

load_dotenv("../.env")

with open("../data/incidents.json") as f:
    incidents = json.load(f)

print("\u2705 Imports ready — WorkflowBuilder, AgentExecutor, Case, Default loaded")

c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


✅ Imports ready — WorkflowBuilder, AgentExecutor, Case, Default loaded


---
## Step 1: Pydantic Models (from Challenge 1)

We reuse the structured output models. Copy your definitions from Challenge 1
or use these minimal versions:

In [2]:
class TriageResult(BaseModel):
    severity: Literal["critical", "high", "medium", "low"]
    is_recurring: bool
    auto_remediation_allowed: bool
    root_cause_hypothesis: str
    recommended_action: str
    escalation_threshold_minutes: int

class DiagnosticsResult(BaseModel):
    root_cause: str
    evidence: list[str]
    affected_components: list[str]
    confidence: float
    recommended_fix: str
    requires_restart: bool

print("\u2705 Models defined")

✅ Models defined


---
## Step 2: Create Agent Factories

Same agents as Challenge 1, wrapped for workflow use.

In [3]:
def make_client() -> FoundryChatClient:
    """Shared client factory — all agents use the same Foundry project."""
    return FoundryChatClient(
        project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
        model=os.environ["FOUNDRY_MODEL"],
        credential=AzureCliCredential(),
    )

def create_triage_agent() -> Agent:
    return Agent(
        client=make_client(),
        name="TriageAgent",
        instructions=(
            "You are an incident Triage Agent. When given an alert:\n"
            "1. Call check_alert_history with the service name\n"
            "2. Call get_runbook with the incident_type\n"
            "Then classify severity and recommend next steps."
        ),
        tools=[check_alert_history, get_runbook],
        default_options=OpenAIChatOptions[Any](response_format=TriageResult),
    )

def create_diagnostics_agent() -> Agent:
    return Agent(
        client=make_client(),
        name="DiagnosticsAgent",
        instructions=(
            "You are an incident Diagnostics Agent. Investigate the root cause:\n"
            "1. Call get_metrics for relevant metric types (memory, latency, cpu, error_rate)\n"
            "2. Call get_logs with severity='error'\n"
            "3. Call check_dependencies to identify cascading failures\n"
            "Provide confidence based on evidence quality."
        ),
        tools=[get_metrics, get_logs, check_dependencies],
        default_options=OpenAIChatOptions[Any](response_format=DiagnosticsResult),
    )

print("\u2705 Agent factories ready")

✅ Agent factories ready


---
## Step 3: Build Workflow Executors

Executors are the **nodes** in your workflow graph. Each receives typed input
and either sends a message downstream or yields workflow output.

### Key Concepts:
- `@executor(id="...")` — decorator to create a workflow node
- `ctx.send_message(data)` — pass data to the next node
- `ctx.yield_output(data)` — emit final workflow result
- `ctx.set_state(key, value)` — store data accessible by any executor
- `ctx.get_state(key)` — retrieve stored data

### Ingestion Executor (provided)

This is the **start node** — receives the raw alert and forwards it to the triage agent.

In [4]:
@executor(id="ingest_alert")
async def ingest_alert(alert_json: str, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
    """Start node: parse the alert and store it in workflow state, then forward to triage."""
    alert = json.loads(alert_json)
    
    # Store the alert in workflow state — accessible by all downstream executors
    ctx.set_state("alert", alert)
    ctx.set_state("service", alert["service"])
    
    # Forward to the triage agent as a user message
    user_msg = Message("user", contents=[
        f"New alert fired:\n"
        f"Title: {alert['title']}\n"
        f"Service: {alert['service']}\n"
        f"Type: {alert['incident_type']}\n"
        f"Description: {alert['description']}"
    ])
    await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))

print("\u2705 ingest_alert executor defined")

✅ ingest_alert executor defined


### Parse Triage Result (provided)

This executor sits **between** the triage agent and the switch-case routing.
It parses the structured JSON, stores results in state, and emits a
routing-friendly dataclass.

In [5]:
@dataclass
class RoutingDecision:
    """Lightweight payload used by edge conditions for routing."""
    severity: str
    service: str


@executor(id="parse_triage")
async def parse_triage(response: AgentExecutorResponse, ctx: WorkflowContext[RoutingDecision]) -> None:
    """Parse triage agent's structured output and emit routing decision."""
    triage = TriageResult.model_validate_json(response.agent_response.text)
    
    # Store full triage result in workflow state for downstream use
    ctx.set_state("triage_result", triage)
    
    # Emit a lightweight routing payload for the switch-case edges
    await ctx.send_message(RoutingDecision(severity=triage.severity, service=ctx.get_state("service")))

print("\u2705 parse_triage executor defined")

✅ parse_triage executor defined


---
## Step 4: Switch-Case Edge Conditions (YOUR TURN)

This is the **key MAF feature** — deterministic routing based on typed data.

You need to write condition functions that inspect the `RoutingDecision` payload
and return `True`/`False` to gate each path.

The pattern:
```python
def get_severity_condition(expected: str):
    def condition(message: Any) -> bool:
        if isinstance(message, RoutingDecision):
            return message.severity == expected
        return False
    return condition
```

In [6]:
from typing import Any

def get_severity_condition(expected: str):
    """Factory: returns a condition that matches RoutingDecision.severity."""
    def condition(message: Any) -> bool:
        if isinstance(message, RoutingDecision):
            return message.severity == expected
        return False
    return condition

is_critical = get_severity_condition("critical")
is_high = get_severity_condition("high")

print("✅ Edge condition factory defined")
print(f"   is_critical(RoutingDecision('critical','svc')): {is_critical(RoutingDecision('critical','svc'))}")
print(f"   is_high(RoutingDecision('high','svc')): {is_high(RoutingDecision('high','svc'))}")
print(f"   is_critical(RoutingDecision('low','svc')): {is_critical(RoutingDecision('low','svc'))}")


✅ Edge condition factory defined
   is_critical(RoutingDecision('critical','svc')): True
   is_high(RoutingDecision('high','svc')): True
   is_critical(RoutingDecision('low','svc')): False


---
## Step 5: Build the Branch Executors (YOUR TURN)

### Critical Path: Forward to Diagnostics Agent

This executor handles the CRITICAL path. It reads triage context from state
and creates a new `AgentExecutorRequest` to send to the diagnostics agent.

In [7]:
@executor(id="to_diagnostics")
async def to_diagnostics(routing: RoutingDecision, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
    """Critical/High path: prepare request for diagnostics agent."""
    triage: TriageResult = ctx.get_state("triage_result")
    service = ctx.get_state("service")
    user_msg = Message("user", contents=[
        f"Investigate this incident:\n"
        f"Service: {service}\n"
        f"Triage hypothesis: {triage.root_cause_hypothesis}\n"
        f"Investigate: {triage.recommended_action}"
    ])
    await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))

print("✅ to_diagnostics executor defined")


✅ to_diagnostics executor defined


### Low-Severity Path: Monitor Only

For LOW severity, we skip diagnostics/remediation entirely and just log + notify.

In [8]:
@executor(id="monitor_only")
async def monitor_only(routing: RoutingDecision, ctx: WorkflowContext[Never, str]) -> None:
    """Low-severity terminal: log and yield output without remediation."""
    alert = ctx.get_state("alert")
    await ctx.yield_output(
        f"📋 LOW severity: {alert['title']} — monitoring only, no action taken."
    )

print("✅ monitor_only executor defined")


✅ monitor_only executor defined


### Communications Executor (provided)

Terminal node for the critical/high path — formats the final incident report.

In [9]:
@executor(id="comms")
async def comms(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Terminal: format diagnostics results into a workflow output report."""
    diag = DiagnosticsResult.model_validate_json(response.agent_response.text)
    service = ctx.get_state("service")
    triage: TriageResult = ctx.get_state("triage_result")
    
    # Store diagnostics for later use
    ctx.set_state("diagnostics_result", diag)
    
    report = (
        f"\U0001f6a8 INCIDENT RESPONSE REPORT\n"
        f"{'='*40}\n"
        f"Service: {service}\n"
        f"Severity: {triage.severity.upper()}\n"
        f"Root Cause: {diag.root_cause}\n"
        f"Confidence: {diag.confidence:.0%}\n"
        f"Affected: {', '.join(diag.affected_components)}\n"
        f"Recommended Fix: {diag.recommended_fix}\n"
        f"Requires Restart: {diag.requires_restart}\n"
    )
    await ctx.yield_output(report)

print("\u2705 comms executor defined")

✅ comms executor defined


---
## Step 6: Wire the Workflow Graph (YOUR TURN)

This is where it all comes together. Use `WorkflowBuilder` to create the DAG:

```
ingest_alert → triage_agent → parse_triage → SWITCH:
    Case("critical") → to_diagnostics → diagnostics_agent → comms
    Case("high")     → to_diagnostics → diagnostics_agent → comms
    Default          → monitor_only
```

### API Reference:
```python
workflow = (
    WorkflowBuilder(start_executor=start_node)
    .add_edge(source, target)                    # Unconditional edge
    .add_edge(source, target, condition=fn)      # Conditional edge
    .add_switch_case_edge_group(source, [        # One-of-N routing
        Case(condition=fn1, target=node_a),
        Case(condition=fn2, target=node_b),
        Default(target=node_c),
    ])
    .build()
)
```

In [11]:
triage_agent_executor = AgentExecutor(create_triage_agent())
diagnostics_agent_executor = AgentExecutor(create_diagnostics_agent())

workflow = (
    WorkflowBuilder(start_executor=ingest_alert)
    .add_edge(ingest_alert, triage_agent_executor)
    .add_edge(triage_agent_executor, parse_triage)
    .add_switch_case_edge_group(parse_triage, [
        Case(condition=is_critical, target=to_diagnostics),
        Case(condition=is_high, target=to_diagnostics),
        Default(target=monitor_only),
    ])
    .add_edge(to_diagnostics, diagnostics_agent_executor)
    .add_edge(diagnostics_agent_executor, comms)
    .build()
)

print("✅ Workflow graph wired!")
print("   critical → to_diagnostics → diagnostics → comms")
print("   high     → to_diagnostics → diagnostics → comms")
print("   default  → monitor_only")


✅ Workflow graph wired!
   critical/high → to_diagnostics → diagnostics → comms
   default       → monitor_only


C:\Users\kiranpanchal\AppData\Local\Temp\ipykernel_3980\1271192069.py:18: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()


---
## Step 7: Run the Workflow

Feed a CRITICAL incident and watch it route through the full pipeline.
Then feed a LOW-severity incident and watch it take the monitor-only path.

In [12]:
# Run with CRITICAL incident (payment-api OOM)
print("\U0001f680 Running workflow with CRITICAL incident...")
print(f"   Alert: {incidents[0]['title']}\n")

events = await workflow.run(json.dumps(incidents[0]))
outputs = events.get_outputs()

if outputs:
    print(outputs[0])
else:
    print("\u274c No output — check your edge conditions and executor logic")

🚀 Running workflow with CRITICAL incident...
   Alert: Payment API P99 Latency > 30s

{"severity":"high","is_recurring":true,"auto_remediation_allowed":true,"root_cause_hypothesis":"Memory overload due to a known connection pool leak, exacerbated by batch job execution.","recommended_action":"Follow the High Latency Response runbook (RB-204): Inspect resource metrics, address the memory leak issue, restart affected pods, and monitor to ensure the latency normalizes.","escalation_threshold_minutes":15}


In [14]:
# Validate the critical path was taken
assert outputs, "Workflow must produce output"
assert "CRITICAL" in outputs[0] or "critical" in outputs[0].lower(), "Should report critical severity"
assert "Root Cause" in outputs[0], "Should include root cause from diagnostics"
print("\u2705 Critical path validation passed")

✅ Critical path validation passed
   Routed through: triage → diagnostics → comms


### Run with LOW-severity incident

In [15]:
# Run with LOW incident (notification-service rate limiting)
print("\U0001f680 Running workflow with LOW-severity incident...")
print(f"   Alert: {incidents[2]['title']}\n")

events_low = await workflow.run(json.dumps(incidents[2]))
outputs_low = events_low.get_outputs()

if outputs_low:
    print(outputs_low[0])
else:
    print("\u274c No output — check the Default path")

🚀 Running workflow with LOW-severity incident...
   Alert: Notification Service Email Delivery Failing

{"severity":"medium","is_recurring":false,"auto_remediation_allowed":false,"root_cause_hypothesis":"SendGrid rate limiting due to overuse; insufficient handling of email dispatch rates under heavy load.","recommended_action":"Follow the Rate Limiting Response runbook (RB-410): Verify SendGrid usage limits, consider rerouting via an alternate provider, and notify vendor support.","escalation_threshold_minutes":20}


In [17]:
# Validate the low-severity path was taken (monitor only, no diagnostics)
assert outputs_low, "Low-severity should still produce output"
assert "monitor" in outputs_low[0].lower() or "LOW" in outputs_low[0], \
    "Low-severity should take the monitor-only path"
print("\u2705 Low-severity routing validation passed")
print("\n\U0001f3af The SAME workflow handles both incidents differently based on severity!")

✅ Low-severity routing validation passed

🎯 The SAME workflow handles both incidents differently based on severity!


---
## \U0001f4aa Stretch Goal: Add the HIGH Path

The current graph routes CRITICAL and HIGH to the same `to_diagnostics` node.
Create a separate HIGH path that skips the full comms report and just posts to Slack.

Hints:
- Create a `@executor(id="comms_lite")` that posts a shorter notification
- Add a separate `Case(condition=get_severity_condition("high"), target=to_diagnostics_high)` 
- Wire the high path to use `comms_lite` instead of `comms`

In [18]:
# Stretch goal: differentiated HIGH vs CRITICAL paths
# ...

---
## \u27a1\ufe0f Next: Challenge 3 — Human-in-the-Loop & Resilience

Your workflow routes correctly but executes autonomously.
In production, you need **human approval** before destructive actions
and **retry logic** when verification fails.

Challenge 3 adds:
- `ctx.request_info()` to pause the workflow and wait for human approval
- `@tool(approval_mode="always_require")` for dangerous operations
- Verification loop with retry-or-escalate logic

[Open Challenge 3 →](../challenge-3/challenge.ipynb)

# Challenge 3: Workflow Orchestration

## From Manual Handoffs to Automated Pipelines

In Challenge 2, you manually passed outputs between agents. Now you'll use MAF's `WorkflowBuilder`
to create an **automated pipeline** with:

- **Sequential execution**: Triage → Diagnostics → Remediation → Verify → Comms
- **Conditional routing**: If verification fails → retry or escalate
- **Type-safe shared state**: A dataclass flows through the pipeline
- **Single entry point**: One call handles the entire incident

```
┌──────────┐    ┌──────────────┐    ┌─────────────┐    ┌──────────┐    ┌───────┐
│  Triage  │───►│ Diagnostics  │───►│ Remediation │───►│  Verify  │───►│ Comms │
└──────────┘    └──────────────┘    └─────────────┘    └────┬─────┘    └───────┘
                                           ▲                │
                                           │    FAIL        │
                                           └────────────────┘
```

---

## How This Challenge Works

1. **Shared State** is provided for you (the `IncidentState` dataclass)
2. **Triage executor** is provided as a reference
3. You build the **Diagnostics**, **Remediation**, and **Verification** executors
4. You implement the **conditional routing logic** (the hard part!)
5. You wire everything together with `WorkflowBuilder`

In [19]:
import os
import sys
import json
from dataclasses import dataclass, field
sys.path.insert(0, "..")
from dotenv import load_dotenv

from agent_framework import WorkflowBuilder, WorkflowContext, executor
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
    restart_pod, scale_service, flush_cache, toggle_feature_flag, escalate_to_human,
    get_health_status, run_smoke_test,
    post_to_slack, create_incident_ticket, update_status_page,
)

load_dotenv("../.env")

with open("../data/incidents.json") as f:
    incidents = json.load(f)

print("\u2705 Imports ready")

ImportError: cannot import name 'AzureAIAgentClient' from 'agent_framework.azure' (c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\azure\__init__.py)

## Step 1: Shared State (Provided)

This dataclass holds everything as it flows through the pipeline.
Each executor reads from state and writes its results back.

In [ ]:
@dataclass
class IncidentState:
    """Shared state that flows through the incident response workflow."""
    # Input (set when workflow starts)
    alert_title: str = ""
    alert_service: str = ""
    alert_severity: str = ""
    alert_description: str = ""
    
    # Results from each stage (filled by executors)
    triage_result: str = ""
    diagnostics_result: str = ""
    remediation_result: str = ""
    verification_result: str = ""
    comms_result: str = ""
    
    # Control flow
    is_resolved: bool = False
    retry_count: int = 0
    max_retries: int = 1

print("\u2705 IncidentState defined")
print("   Fields:", [f.name for f in IncidentState.__dataclass_fields__.values()])

## Step 2: Triage Executor (REFERENCE)

Study this executor. Key things to notice:
1. `@executor` decorator marks it as a workflow step
2. `ctx: WorkflowContext[IncidentState]` gives access to shared state
3. It reads from state (`ctx.state.alert_title`)
4. It writes to state (`state.triage_result = result.text`)
5. It returns the **name of the next step** as a string

### Expected Output
```
============================================================
📋 TRIAGE STAGE
============================================================
[Agent calls check_alert_history, get_runbook]
Triage summary: Critical, recurring, auto-fix allowed...
→ Next: diagnostics
```

In [ ]:
@executor
async def triage_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """REFERENCE: First step — triage the incoming alert."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"\U0001f4cb TRIAGE STAGE")
    print(f"{'='*60}")
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="TriageAgent",
            instructions="""You are an incident Triage Agent. When given an alert:
1. Call check_alert_history with the service name
2. Call get_runbook with the incident type
Output: severity, recurring status, auto-remediation allowed, next steps.
Be concise and factual.""",
            tools=[check_alert_history, get_runbook],
        )
        
        result = await agent.run(
            f"Alert: {state.alert_title}\nService: {state.alert_service}\n"
            f"Severity: {state.alert_severity}\nDescription: {state.alert_description}"
        )
        
        state.triage_result = result.text
        print(f"\n{result.text[:500]}")
    
    return "diagnostics"  # Next step name

## Step 3: Diagnostics Executor (YOUR TURN)

Build the diagnostics executor. It should:
1. Read triage results from `ctx.state.triage_result`
2. Create a diagnostics agent with `get_metrics`, `get_logs`, `check_dependencies`
3. Pass both triage context and alert info in the prompt
4. Store the result in `state.diagnostics_result`
5. Return `"remediation"` as the next step

### Expected Output
```
============================================================
🔍 DIAGNOSTICS STAGE
============================================================
Root cause: pod-3 OOM, memory at 3891MB/4096MB, 4 restarts...
→ Next: remediation
```

In [ ]:
@executor
async def diagnostics_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """YOUR CODE: Diagnose the root cause."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"\U0001f50d DIAGNOSTICS STAGE")
    print(f"{'='*60}")
    
    # ╔══════════════════════════════════════════════════════════════╗
    # ║  YOUR CODE HERE                                             ║
    # ║  1. Create AzureCliCredential + AzureAIAgentClient          ║
    # ║  2. Create agent with tools: get_metrics, get_logs,         ║
    # ║     check_dependencies                                      ║
    # ║  3. Run agent with triage context + alert service           ║
    # ║  4. Store result in state.diagnostics_result                ║
    # ╚══════════════════════════════════════════════════════════════╝
    
    # TODO: Your implementation here
    pass
    
    return "remediation"

## Step 4: Remediation Executor (YOUR TURN)

Build the remediation executor. It should:
1. Read diagnostics results from `ctx.state.diagnostics_result`
2. Create agent with `restart_pod`, `scale_service`, `flush_cache`, `toggle_feature_flag`, `escalate_to_human`
3. Take action based on the root cause
4. Store result in `state.remediation_result`
5. Return `"verification"` as the next step

### Expected Output
```
============================================================
🔧 REMEDIATION STAGE (attempt 1)
============================================================
Action: Restarted payment-api-pod-3
Result: Success — pod restarted, new instance healthy
→ Next: verification
```

In [ ]:
@executor
async def remediation_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """YOUR CODE: Execute the fix."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"\U0001f527 REMEDIATION STAGE (attempt {state.retry_count + 1})")
    print(f"{'='*60}")
    
    # ╔══════════════════════════════════════════════════════════════╗
    # ║  YOUR CODE HERE                                             ║
    # ║  Same pattern as triage_executor but with fix tools         ║
    # ║  Store result in state.remediation_result                   ║
    # ╚══════════════════════════════════════════════════════════════╝
    
    # TODO: Your implementation here
    pass
    
    return "verification"

## Step 5: Verification Executor with ROUTING LOGIC (YOUR TURN — HARDEST PART)

This is the critical executor. Unlike the others, it has **conditional routing**:

```
if fix worked (RESOLVED):
    → go to "communications"     # Success path
elif retries remaining:
    → go to "remediation"         # Retry the fix
else:
    → go to "communications"      # Give up, notify team
```

### Requirements
1. Create verification agent with `get_health_status` + `run_smoke_test`
2. Tell the agent to end its response with `VERDICT: RESOLVED` or `VERDICT: FAILED`
3. Parse the verdict from the response
4. Update `state.is_resolved` and `state.retry_count`
5. Return the correct next step based on the routing logic above

### Expected Output (success path)
```
============================================================
✅ VERIFICATION STAGE
============================================================
Health: All pods healthy, latency 180ms
Smoke tests: 5/5 passing
VERDICT: RESOLVED
→ Next: communications (success)
```

In [ ]:
@executor
async def verification_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """YOUR CODE: Verify the fix AND implement routing logic."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"\u2705 VERIFICATION STAGE")
    print(f"{'='*60}")
    
    # ╔══════════════════════════════════════════════════════════════╗
    # ║  YOUR CODE HERE                                             ║
    # ║  1. Create verification agent (health + smoke test tools)   ║
    # ║  2. Tell it to end with VERDICT: RESOLVED or VERDICT: FAILED║
    # ║  3. Store result in state.verification_result               ║
    # ║  4. Parse the verdict and implement routing:                ║
    # ║     - RESOLVED → set state.is_resolved=True, return        ║
    # ║       "communications"                                      ║
    # ║     - FAILED + retries left → increment retry_count,       ║
    # ║       return "remediation"                                  ║
    # ║     - FAILED + no retries → return "communications"         ║
    # ╚══════════════════════════════════════════════════════════════╝
    
    # TODO: Your implementation here
    pass
    
    return "communications"  # Replace with your routing logic

## Step 6: Communications Executor (Provided)

This is the final step — it always returns `None` to end the workflow.

In [ ]:
@executor
async def communications_executor(ctx: WorkflowContext[IncidentState]) -> None:
    """Final step: Notify the team. Returns None to end workflow."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"\U0001f4e2 COMMUNICATIONS STAGE")
    print(f"{'='*60}")
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="CommunicationsAgent",
            instructions="""You are the Communications Agent. Notify the team.
1. Post to Slack (#incidents) — brief 4-line summary
2. Create an incident ticket with the full timeline
3. Update status page to 'operational' if resolved, 'degraded' if not""",
            tools=[post_to_slack, create_incident_ticket, update_status_page],
        )
        
        status = "RESOLVED" if state.is_resolved else "ESCALATED (auto-fix failed)"
        result = await agent.run(
            f"Status: {status}\n"
            f"Service: {state.alert_service}\n"
            f"Triage: {state.triage_result[:200]}\n"
            f"Diagnostics: {state.diagnostics_result[:200]}\n"
            f"Remediation: {state.remediation_result[:200]}\n"
            f"Verification: {state.verification_result[:200]}"
        )
        
        state.comms_result = result.text
        print(f"\n{result.text[:500]}")
    
    return None  # End of workflow

## Step 7: Wire It Together (YOUR TURN)

Use `WorkflowBuilder` to register all executors and build the workflow.

```python
workflow = (
    WorkflowBuilder[IncidentState]()
    .add_executor(executor_function, name="step_name")
    # ... add more
    .build()
)
```

The `name` parameter must match what executors return (e.g., triage returns `"diagnostics"`,
so the diagnostics executor must be registered with `name="diagnostics"`).

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  YOUR CODE: Build the workflow                              ║
# ║  Register all 5 executors with the correct names:           ║
# ║  triage, diagnostics, remediation, verification, comms      ║
# ╚══════════════════════════════════════════════════════════════╝

workflow = (
    WorkflowBuilder[IncidentState]()
    # TODO: Add all 5 executors here
    .build()
)

print("\u2705 Workflow built!")

## Step 8: Run It!

### Expected Output
```
🚨 INCIDENT RESPONSE INITIATED
   Alert: High Latency on payment-api
   Service: payment-api

============================================================
📋 TRIAGE STAGE
============================================================
...
============================================================
🔍 DIAGNOSTICS STAGE
============================================================
...
============================================================
🔧 REMEDIATION STAGE (attempt 1)
============================================================
...
============================================================
✅ VERIFICATION STAGE
============================================================
...
============================================================
📢 COMMUNICATIONS STAGE
============================================================
...

============================================================
🏁 INCIDENT RESPONSE COMPLETE
============================================================
   Resolved: ✅ Yes
   Retries: 0
   Stages completed: 5
```

In [ ]:
# Run the workflow on incident #1 (payment-api)
alert = incidents[0]

print(f"\U0001f6a8 INCIDENT RESPONSE INITIATED")
print(f"   Alert: {alert['title']}")
print(f"   Service: {alert['service']}")
print(f"   Severity: {alert['severity']}")

initial_state = IncidentState(
    alert_title=alert["title"],
    alert_service=alert["service"],
    alert_severity=alert["severity"],
    alert_description=alert["description"],
)

final_state = await workflow.run(state=initial_state)

print(f"\n\n{'='*60}")
print(f"\U0001f3c1 INCIDENT RESPONSE COMPLETE")
print(f"{'='*60}")
print(f"   Resolved: {'\u2705 Yes' if final_state.is_resolved else '\u274c No (escalated)'}")
print(f"   Retries: {final_state.retry_count}")
print(f"   Stages completed: {sum(1 for r in [final_state.triage_result, final_state.diagnostics_result, final_state.remediation_result, final_state.verification_result, final_state.comms_result] if r)}")

In [ ]:
# ✅ VALIDATION
checks = {
    "Triage completed": bool(final_state.triage_result),
    "Diagnostics completed": bool(final_state.diagnostics_result),
    "Remediation completed": bool(final_state.remediation_result),
    "Verification completed": bool(final_state.verification_result),
    "Communications completed": bool(final_state.comms_result),
    "Incident resolved": final_state.is_resolved,
}

print("\n\U0001f9ea WORKFLOW VALIDATION:")
for check, passed in checks.items():
    print(f"{'\u2705' if passed else '\u274c'} {check}")

if all(checks.values()):
    print("\n\U0001f389 Workflow executed perfectly!")
else:
    print("\n\u26a0\ufe0f Some stages didn't complete. Check your executor implementations.")

## Step 9: Prove It's Adaptive — Run Incident #3

The SAME workflow should make DIFFERENT decisions on a different incident.
Incident #3 is a notification-service email failure — the fix is toggling a feature flag, not restarting a pod.

### Expected Difference
| | Incident #1 (payment-api) | Incident #3 (notification-service) |
|---|---|---|
| Root cause | OOM on pod-3 | Email provider rate limiting |
| Remediation | `restart_pod` | `toggle_feature_flag` |
| Target | payment-api-pod-3 | use_backup_email → true |

In [ ]:
# Run on incident #3
alert3 = incidents[2]

print(f"\U0001f6a8 INCIDENT #3: {alert3['title']}")
print(f"   Service: {alert3['service']} | Severity: {alert3['severity']}")
print(f"   Description: {alert3['description']}")

state3 = IncidentState(
    alert_title=alert3["title"],
    alert_service=alert3["service"],
    alert_severity=alert3["severity"],
    alert_description=alert3["description"],
)

final_state3 = await workflow.run(state=state3)

print(f"\n\n{'='*60}")
print(f"\U0001f3c1 INCIDENT #3 COMPLETE")
print(f"{'='*60}")
print(f"   Resolved: {'\u2705' if final_state3.is_resolved else '\u274c'}")

# Check different decisions were made
print(f"\n\U0001f9ea DIFFERENT DECISIONS?")
remed_lower = final_state3.remediation_result.lower()
checks = {
    "Used toggle/feature_flag (not restart)": any(w in remed_lower for w in ["toggle", "feature", "flag", "backup"]),
    "Different root cause than incident #1": "oom" not in final_state3.diagnostics_result.lower() or "rate" in final_state3.diagnostics_result.lower(),
}
for check, passed in checks.items():
    print(f"{'\u2705' if passed else '\u274c'} {check}")

---
## What You Achieved

| Feature | Challenge 2 (Manual) | Challenge 3 (Workflow) |
|---------|---------------------|------------------------|
| Agent coordination | Manual copy-paste | Automated pipeline |
| Error handling | None | Retry + escalation |
| Routing | Fixed sequence | Conditional (verify → retry or complete) |
| State management | Ad-hoc variables | Typed `IncidentState` dataclass |
| Reusability | One-off scripts | Same workflow handles ANY incident |
| Entry point | 5 separate function calls | One `workflow.run()` call |

---

## \u27a1\ufe0f Next: Challenge 4 — Memory Patterns

The workflow works but starts from scratch every time. A human SRE remembers past incidents.
In Challenge 4, you'll add memory so the system gets **faster** with experience.

# 🔄 Step 3: Workflow Orchestration

## From Manual Handoffs to Automated Pipelines

In Step 2, we manually passed outputs between agents. Now we'll use MAF's `WorkflowBuilder`
to create an **automated incident response pipeline** with:

- **Sequential execution**: Triage → Diagnostics → Remediation → Verify → Comms
- **Conditional routing**: If verification fails → retry or escalate
- **Type-safe state**: Shared context flows through the pipeline
- **Single entry point**: One call handles the entire incident

---

## Architecture

```
┌──────────┐    ┌──────────────┐    ┌─────────────┐    ┌──────────┐    ┌───────┐
│  Triage  │───►│ Diagnostics  │───►│ Remediation │───►│  Verify  │───►│ Comms │
└──────────┘    └──────────────┘    └─────────────┘    └────┬─────┘    └───────┘
                                           ▲                │
                                           │    FAIL        │
                                           └────────────────┘
```

In [ ]:
import os
import sys
import json
from dataclasses import dataclass, field
sys.path.insert(0, "..")
from dotenv import load_dotenv

from agent_framework import WorkflowBuilder, WorkflowContext, executor
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
    restart_pod, scale_service, flush_cache, toggle_feature_flag, escalate_to_human,
    get_health_status, run_smoke_test,
    post_to_slack, create_incident_ticket, update_status_page,
)

load_dotenv("../.env")

# Load incident data
with open("../data/incidents.json") as f:
    incidents = json.load(f)

print("✅ Imports ready")

## Define Shared State

The workflow needs shared state to pass information between executors.
We use a `dataclass` to hold the incident context as it flows through the pipeline.

In [ ]:
@dataclass
class IncidentState:
    """Shared state that flows through the incident response workflow."""
    # Input
    alert_title: str = ""
    alert_service: str = ""
    alert_severity: str = ""
    alert_description: str = ""
    
    # Accumulated context from each stage
    triage_result: str = ""
    diagnostics_result: str = ""
    remediation_result: str = ""
    verification_result: str = ""
    comms_result: str = ""
    
    # Control flow
    is_resolved: bool = False
    retry_count: int = 0
    max_retries: int = 1

print("✅ IncidentState defined")

## Define Workflow Executors

Each executor wraps one of our specialized agents. The `@executor` decorator
tells MAF this is a step in the workflow.

### 🎯 YOUR TASK
Complete the `triage_executor` and `diagnostics_executor` below.
The remediation, verification, and comms executors are provided as reference.

In [ ]:
@executor
async def triage_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """First step: Triage the incoming alert."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"📋 TRIAGE STAGE")
    print(f"{'='*60}")
    
    # TODO: Create the triage agent with check_alert_history and get_runbook tools
    # TODO: Run it with the alert details from state
    # TODO: Store result in state.triage_result
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="TriageAgent",
            instructions="""You are an incident Triage Agent. When given an alert:
1. Use check_alert_history to check if this is a recurring issue
2. Use get_runbook to find the remediation playbook
3. Output: severity, recurring status, whether auto-remediation is allowed, recommended incident type for runbook lookup.
Be concise.""",
            tools=[check_alert_history, get_runbook],
        )
        
        result = await agent.run(
            f"Alert: {state.alert_title}\nService: {state.alert_service}\n"
            f"Severity: {state.alert_severity}\nDescription: {state.alert_description}"
        )
        
        state.triage_result = result.text
        print(f"\n{result.text}")
    
    return "diagnostics"  # Next step

In [ ]:
@executor
async def diagnostics_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """Second step: Diagnose root cause."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"🔍 DIAGNOSTICS STAGE")
    print(f"{'='*60}")
    
    # TODO: Create the diagnostics agent with get_metrics, get_logs, check_dependencies
    # TODO: Include triage context in the prompt
    # TODO: Store result in state.diagnostics_result
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="DiagnosticsAgent",
            instructions="""You are a Diagnostics Agent. Investigate the incident using your tools.
1. Use get_metrics to check resource utilization
2. Use get_logs to find error patterns
3. Use check_dependencies to identify cascading failures
Output: specific root cause with evidence, affected components, and exact remediation action needed.""",
            tools=[get_metrics, get_logs, check_dependencies],
        )
        
        result = await agent.run(
            f"Triage summary:\n{state.triage_result}\n\n"
            f"Service: {state.alert_service}\nInvestigate root cause."
        )
        
        state.diagnostics_result = result.text
        print(f"\n{result.text}")
    
    return "remediation"  # Next step

In [ ]:
@executor
async def remediation_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """Third step: Execute the fix."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"🔧 REMEDIATION STAGE (attempt {state.retry_count + 1})")
    print(f"{'='*60}")
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="RemediationAgent",
            instructions="""You are a Remediation Agent. Execute the fix based on diagnostics.
Use the appropriate tool based on the root cause:
- OOM/memory leak → restart_pod
- High CPU/traffic → scale_service
- Stale cache → flush_cache
- External dependency failure → toggle_feature_flag for failover
- Cannot auto-fix → escalate_to_human
Use exact names from the diagnostics (specific pod names, service names, flag names).""",
            tools=[restart_pod, scale_service, flush_cache, toggle_feature_flag, escalate_to_human],
        )
        
        result = await agent.run(
            f"Diagnostics findings:\n{state.diagnostics_result}\n\n"
            f"Service: {state.alert_service}\nExecute the appropriate fix."
        )
        
        state.remediation_result = result.text
        print(f"\n{result.text}")
    
    return "verification"  # Next step

In [ ]:
@executor
async def verification_executor(ctx: WorkflowContext[IncidentState]) -> str:
    """Fourth step: Verify the fix worked. Routes to comms or back to remediation."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"✅ VERIFICATION STAGE")
    print(f"{'='*60}")
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="VerificationAgent",
            instructions="""You are a Verification Agent. Check if the fix worked.
1. Use get_health_status to check service health
2. Use run_smoke_test to run functional tests
End your response with exactly one of: VERDICT: RESOLVED or VERDICT: FAILED""",
            tools=[get_health_status, run_smoke_test],
        )
        
        result = await agent.run(
            f"Remediation performed:\n{state.remediation_result}\n\n"
            f"Service: {state.alert_service}\nVerify the fix worked."
        )
        
        state.verification_result = result.text
        print(f"\n{result.text}")
        
        # ROUTING LOGIC: Check if resolved or needs retry
        if "VERDICT: RESOLVED" in result.text.upper() or "RESOLVED" in result.text.upper():
            state.is_resolved = True
            return "communications"  # Success path
        else:
            state.retry_count += 1
            if state.retry_count >= state.max_retries:
                print("\n⚠️ Max retries reached — escalating to human")
                return "communications"  # Give up, notify team
            return "remediation"  # Retry the fix

In [ ]:
@executor
async def communications_executor(ctx: WorkflowContext[IncidentState]) -> None:
    """Final step: Notify team and create records."""
    state = ctx.state
    print(f"\n{'='*60}")
    print(f"📢 COMMUNICATIONS STAGE")
    print(f"{'='*60}")
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        agent = client.create_agent(
            name="CommunicationsAgent",
            instructions="""You are the Communications Agent. Notify the team about the incident resolution.
1. Post a concise summary to Slack (#incidents channel) with severity based on resolution status
2. Create an incident ticket with the full timeline
3. Update the status page to 'operational' if resolved, 'degraded' if not
Keep the Slack message brief (4-5 lines max).""",
            tools=[post_to_slack, create_incident_ticket, update_status_page],
        )
        
        status = "RESOLVED" if state.is_resolved else "ESCALATED (auto-fix failed)"
        
        result = await agent.run(
            f"Incident status: {status}\n\n"
            f"Timeline:\n- Triage: {state.triage_result[:200]}\n"
            f"- Diagnostics: {state.diagnostics_result[:200]}\n"
            f"- Remediation: {state.remediation_result[:200]}\n"
            f"- Verification: {state.verification_result[:200]}\n\n"
            f"Service: {state.alert_service}\n"
            f"Notify the team."
        )
        
        state.comms_result = result.text
        print(f"\n{result.text}")
    
    return None  # End of workflow

## Build and Run the Workflow

### 🎯 YOUR TASK
Wire the executors together using `WorkflowBuilder`. The flow is:

```
triage → diagnostics → remediation → verification → communications
                                 ▲         │
                                 └─ FAIL ──┘
```

In [ ]:
# Build the workflow
workflow = (
    WorkflowBuilder[IncidentState]()
    .add_executor(triage_executor, name="triage")
    .add_executor(diagnostics_executor, name="diagnostics")
    .add_executor(remediation_executor, name="remediation")
    .add_executor(verification_executor, name="verification")
    .add_executor(communications_executor, name="communications")
    .build()
)

print("✅ Workflow built!")
print("   Flow: triage → diagnostics → remediation → verification → communications")
print("   Retry: verification can loop back to remediation if fix fails")

In [ ]:
# Run the workflow on incident #1 (payment-api high latency)
alert = incidents[0]

print(f"🚨 INCIDENT RESPONSE INITIATED")
print(f"   Alert: {alert['title']}")
print(f"   Service: {alert['service']}")
print(f"   Severity: {alert['severity']}")
print(f"\n{'='*60}")

# Initialize state with the alert data
initial_state = IncidentState(
    alert_title=alert["title"],
    alert_service=alert["service"],
    alert_severity=alert["severity"],
    alert_description=alert["description"],
)

# Run the full pipeline
final_state = await workflow.run(state=initial_state)

print(f"\n\n{'='*60}")
print(f"🏁 INCIDENT RESPONSE COMPLETE")
print(f"{'='*60}")
print(f"   Resolved: {'✅ Yes' if final_state.is_resolved else '❌ No (escalated)'}")
print(f"   Retries: {final_state.retry_count}")

## 🧪 Try a Different Incident

Run the same workflow on incident #3 (notification-service email failures).
Notice how the agents make **different decisions** — toggle feature flag instead of restart.

In [ ]:
# Try incident #3: notification-service email failures
alert2 = incidents[2]

print(f"🚨 INCIDENT RESPONSE INITIATED")
print(f"   Alert: {alert2['title']}")
print(f"   Service: {alert2['service']}")
print(f"   Severity: {alert2['severity']}")

state2 = IncidentState(
    alert_title=alert2["title"],
    alert_service=alert2["service"],
    alert_severity=alert2["severity"],
    alert_description=alert2["description"],
)

final_state2 = await workflow.run(state=state2)

print(f"\n\n🏁 COMPLETE — Resolved: {'✅' if final_state2.is_resolved else '❌'}")

## 🎉 What We Achieved

With workflow orchestration, we now have:

| Feature | Before (Step 2) | After (Step 3) |
|---------|----------------|----------------|
| Agent coordination | Manual copy-paste | Automated pipeline |
| Error handling | None | Retry + escalation |
| Routing | Fixed sequence | Conditional (verify → retry or complete) |
| State management | Ad-hoc variables | Typed `IncidentState` dataclass |
| Reusability | One-off scripts | Same workflow handles any incident |

---

## ➡️ Next: Step 4 — Memory Patterns

The workflow works, but it doesn't **learn**. Every incident starts from scratch.
In Step 4, we'll add memory so the system:
- Remembers past incidents and their resolutions
- Gets faster at diagnosing recurring issues
- Builds institutional knowledge over time